# f2–f4 统计汇总

`docs/experiments/2026-10-27-f2-f4-summary.md`（下称汇总文档）中由原始数据补算的数都出自本 notebook。各组的定义、检查点与题量见汇总文档第 1 节；各实验记录见该文档开头的出处表。

- **数据**：只读 `~/pw-lab/runs/` 下的原始数据，不调用推理服务。
  - 写者：`f2/sessions.jsonl`、`f3/sessions.jsonl` 与其中登记的事件流（`runs/*.jsonl`），A4 另读每行的 `graphiti` 字段；
  - 读者：`ref1/answers/{P,C,D}/`、`f3/answers/{A3,A2,A0,A4,A3F}/` 下的 `k<K>.jsonl` 与 `k<K>-aux.jsonl`，每条记录自带用时、`usage` 与工具次数；
  - 准确率：现行计分 `f3/grades/scores/`、`ref1/grades/scores/`；修正前的计分 `f3/grades/scores.before-t3fix-1025T0734/` 与 ref1 的冻结快照 `ref1/frozen/{P-20261022,C-20261021,D-20261021}/grades/scores/`。
- **只能从记录文档取的数**（墙钟、各记录给出的合计）以常量写入，注明出处文件与节号；合计只用于核对（第 1 节）。
- **口径**：token 为各次模型调用之和，pi 的 input 不含命中缓存的部分（未缓存输入 $I_u$），缓存读 $C_r$ 另计，输出 $O$ 含思考。总 token $T = I_u + C_r + O$；加权成本 $W_{\alpha,\beta} = I_u + \alpha C_r + \beta O$（等价未缓存输入 token）。A4 的 Graphiti 内部 LLM 只有 prompt $P_g$ 与 completion $Q_g$，prompt 全按未缓存计：$W^{\mathrm{A4}} = W_{\alpha,\beta} + P_g + \beta Q_g$。加权只是相对口径，不是任何服务商的账单（本实验为自部署的 Qwen 27B，没有计费）。
- 本 notebook 只做描述性统计，不做新的检验，不重算置信区间（区间见各实验记录）。

## 配置

In [ ]:
import json
from pathlib import Path

import pandas as pd

RUNS = Path.home() / "pw-lab" / "runs"  # 原始数据根目录（仓库外）

# 加权参数 (α, β)：I_u : C_r : O = 1 : α : β
WEIGHTS = {"main": (0.1, 5), "sens": (0.25, 4)}

KS = [10, 20, 30]
BOUNDS = {10: 19, 20: 46, 30: 70}  # 检查点 k10、k20、k30 在第 19、46、70 个写者会话之后
WRITERS = [("f2", "A3", "f2-A3"), ("f3", "A3", "A3"), ("f3", "A2", "A2"), ("f3", "A0", "A0"), ("f3", "A4", "A4")]
READERS = [("ref1", "P", "P", KS), ("ref1", "C", "C", [30]), ("ref1", "D", "D", [30]),
           ("f3", "A3", "A3", KS), ("f3", "A2", "A2", KS), ("f3", "A0", "A0", KS),
           ("f3", "A4", "A4", KS), ("f3", "A3F", "A3F", KS)]
COMPARED = ["P", "A3", "A2", "A0", "A4", "A3F"]  # 进成本比较的读者组；C、D 为 12 路并发，只作参照
BUILD_OF = {"P": None, "A3": "A3", "A2": "A2", "A0": "A0", "A4": "A4", "A3F": "A3"}  # A3F 读 A3 的库
CROSS, SINGLE = ["T3", "T7", "T8"], ["T9", "T10", "T11", "T14"]  # 题型族（f4 设计第 3 节）

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

## 0 读入原始数据

写者：每个会话的事件流中，`type == message_end` 且角色为 assistant 的消息即一次模型调用，`usage` 的 input、cacheRead、output 求和，`content` 中 `toolCall` 的条数为工具调用；与 `pweval.harness.timeline.usage`、`tool_counts` 同一规则。读者：回答记录中的 `seconds`、`usage`、`tools` 求和。

In [ ]:
def assistant_messages(path):
    out = []
    for line in path.read_text(encoding="utf-8").splitlines():
        try:
            e = json.loads(line)
        except json.JSONDecodeError:
            continue
        m = e.get("message") or {}
        if e.get("type") == "message_end" and m.get("role") == "assistant":
            out.append(m)
    return out


def usage_sum(messages, key):
    return sum(int((m.get("usage") or {}).get(key) or 0) for m in messages)


rows = []
for run, arm, name in WRITERS:
    for r in map(json.loads, open(RUNS / run / "sessions.jsonl")):
        if r["arm"] != arm:
            continue
        ms = assistant_messages(RUNS / run / r["events"])
        g = r.get("graphiti") or {}
        rows.append({
            "group": name, "seq": r["seq"], "type": r["type"],
            "k": next(K for K, b in BOUNDS.items() if r["seq"] <= b),
            "seconds": r["seconds"], "calls": len(ms),
            "tools": sum(1 for m in ms for c in m.get("content", []) if c.get("type") == "toolCall"),
            "input": usage_sum(ms, "input"), "cacheRead": usage_sum(ms, "cacheRead"), "output": usage_sum(ms, "output"),
            "g_calls": g.get("calls", 0), "g_prompt": g.get("prompt", 0), "g_completion": g.get("completion", 0),
            "g_settle_s": g.get("settle_s", 0.0),
        })
writers = pd.DataFrame(rows)
assert all(sorted(s) == list(range(1, 71)) for _, s in writers.groupby("group").seq)

rows = []
for run, arm, name, ks in READERS:
    for k in ks:
        for part, fn in (("main", f"k{k}.jsonl"), ("aux", f"k{k}-aux.jsonl")):
            for r in map(json.loads, open(RUNS / run / "answers" / arm / fn)):
                u = r.get("usage") or {}
                rows.append({
                    "group": name, "k": k, "part": part, "probe": r["probe"], "template": r.get("template"),
                    "seconds": r.get("seconds") or 0.0, "exit": r.get("exit"), "stop": r.get("stop"),
                    "calls": u.get("calls", 0), "tools": sum((r.get("tools") or {}).values()),
                    "input": u.get("input", 0), "cacheRead": u.get("cacheRead", 0), "output": u.get("output", 0),
                })
readers = pd.DataFrame(rows)
print(writers.groupby("group").size().to_dict())
print(readers.groupby(["group", "k", "part"]).size().unstack().fillna(0).astype(int))

In [ ]:
RAW = ["seconds", "calls", "tools", "input", "cacheRead", "output"]
G = ["g_calls", "g_prompt", "g_completion", "g_settle_s"]


def add_costs(df):
    # 总 token、两种加权成本；有内部 LLM 列时另给含内部 LLM 的值
    df = df.copy()
    df["T"] = df.input + df.cacheRead + df.output
    for key, (a, b) in WEIGHTS.items():
        df[f"W_{key}"] = df.input + a * df.cacheRead + b * df.output
        if "g_prompt" in df:
            df[f"Wg_{key}"] = df[f"W_{key}"] + df.g_prompt + b * df.g_completion
    if "g_prompt" in df:
        df["Tg"] = df["T"] + df.g_prompt + df.g_completion
    df["hours"] = df.seconds / 3600
    return df


def in_millions(df):
    # 展示用：token 与加权成本以百万（M）计，保留两位小数
    out = df.copy()
    for c in out.columns:
        if c in ("input", "cacheRead", "output", "T", "Tg", "g_prompt", "g_completion") or c.startswith("W"):
            out[c] = (out[c] / 1e6).round(2)
    if "hours" in out:
        out["hours"] = out.hours.round(2)
    return out.drop(columns=[c for c in ("seconds", "g_settle_s") if c in out])

## 1 与记录核对

各记录给出的合计以常量写入，原始数据算出的值须与之相同；不同即报错。用时只核对到记录给出的小数位。

In [ ]:
# (用时小时, 模型调用, 工具调用, 未缓存输入, 缓存读, 输出)
RECORD_WRITERS = {
    "f2-A3": (10.28, 1827, 2375, 7597600, 160050624, 3292294),  # f2/2026-10-21-f2-A3-writers.md §5
    "A3": (10.54, 1880, 2427, 8138462, 164715200, 3310921),     # f3/write/2026-10-22-f3-A3-writers.md §5
    "A2": (9.55, 1803, 2447, 7677389, 153739648, 3028502),      # f3/write/2026-10-23-f3-A2-writers.md §5
    "A0": (16.11, 3407, 5016, 9367182, 364725568, 5269322),     # f3/write/2026-10-22-f3-A0-writers.md §5
    "A4": (28.33, 3212, 4339, 21455900, 351011546, 5663149),    # f3/write/2026-10-23-f3-A4-writers.md §5
}
RECORD_A4_GRAPHITI = (12509, 48899529, 2350208, 1.49)  # 调用、prompt、completion、settle 小时；A4 写者记录 §5
RECORD_READERS = {
    "P": (38.8, 8169, 9043, 37476937, 515513997, 6217699),     # f3/read/2026-10-24-f3-A3-readers.md §4.2
    "A3": (43.1, 5315, 6753, 36565635, 356100326, 7167163),    # 同上
    "A0": (31.9, 5802, 7677, 24614229, 281045801, 6371383),    # f3/read/2026-10-24-f3-A0-readers.md §5.1
    "A2": (45.6, 5802, 7185, 39025321, 397073905, 7554515),    # f3/read/2026-10-25-f3-A2-readers.md §5.1
    "A4": (93.5, 9618, 13682, 86324034, 790929411, 11876055),  # f3/read/2026-10-25-f3-A4-readers.md §5.1
}
RECORD_A3F_HOURS = 37.2  # f4/2026-10-26-f4-A3F-readers.md §4（记录只给用时合计）


def check(df, record, digits):
    tot = df.groupby("group")[RAW].sum()
    for g, (h, *ints) in record.items():
        got = tot.loc[g]
        assert round(got.seconds / 3600, digits) == h, (g, got.seconds / 3600, h)
        assert [int(got[c]) for c in RAW[1:]] == ints, (g, [int(got[c]) for c in RAW[1:]], ints)


check(writers, RECORD_WRITERS, 2)
check(readers, RECORD_READERS, 1)
a4 = writers[writers.group == "A4"]
assert (a4.g_calls.sum(), a4.g_prompt.sum(), a4.g_completion.sum()) == RECORD_A4_GRAPHITI[:3]
assert round(a4.g_settle_s.sum() / 3600, 2) == RECORD_A4_GRAPHITI[3]
assert round(readers[readers.group == "A3F"].seconds.sum() / 3600, 1) == RECORD_A3F_HOURS
print("写者与读者的合计都与记录相同")

In [ ]:
# 写者各检查点区间的值，与各写者记录 §5“按检查点区间”的表相同（用时小时、模型调用、未缓存输入、缓存读、输出）
RECORD_WRITER_INTERVALS = {  # 出处同 RECORD_WRITERS
    ("f2-A3", 10): (2.13, 421, 1500206, 31204992, 711968), ("f2-A3", 20): (4.50, 734, 3347809, 65871168, 1432679),
    ("f2-A3", 30): (3.64, 672, 2749585, 62974464, 1147647),
    ("A3", 10): (2.23, 430, 1520899, 33289664, 751943), ("A3", 20): (3.89, 705, 3056749, 59866048, 1225928),
    ("A3", 30): (4.42, 745, 3560814, 71559488, 1333050),
    ("A2", 10): (2.18, 377, 1533041, 27677888, 742974), ("A2", 20): (3.89, 733, 2966704, 61608320, 1198139),
    ("A2", 30): (3.48, 693, 3177644, 64453440, 1087389),
    ("A0", 10): (3.07, 623, 1821432, 55446912, 1041239), ("A0", 20): (6.23, 1412, 3542183, 152844288, 2022371),
    ("A0", 30): (6.81, 1372, 4003567, 156434368, 2205712),
    ("A4", 10): (6.28, 750, 4814222, 83452288, 1233647), ("A4", 20): (11.40, 1277, 9047998, 135999194, 2455333),
    ("A4", 30): (10.65, 1185, 7593680, 131560064, 1974169),
}
iv = writers.groupby(["group", "k"])[RAW].sum()
for (g, k), (h, calls, i, c, o) in RECORD_WRITER_INTERVALS.items():
    row = iv.loc[(g, k)]
    assert round(row.seconds / 3600, 2) == h and (row.calls, row.input, row.cacheRead, row.output) == (calls, i, c, o), (g, k)
print("写者各区间与记录相同")

## 2 准确率

现行计分已含 T3 look-back 的修正（eval `765e920`）。修正前：f3 为 `scores.before-t3fix-1025T0734/`，P、C、D 为冻结快照。C、D 与检查点无关，k10、k20 的值取 P 在该检查点的同一组题。

In [ ]:
SCORE_NOW = {"ref1": RUNS / "ref1/grades/scores", "f3": RUNS / "f3/grades/scores"}
SCORE_PRE = {
    "A3": RUNS / "f3/grades/scores.before-t3fix-1025T0734", "A0": RUNS / "f3/grades/scores.before-t3fix-1025T0734",
    "P": RUNS / "ref1/frozen/P-20261022/grades/scores", "C": RUNS / "ref1/frozen/C-20261021/grades/scores",
    "D": RUNS / "ref1/frozen/D-20261021/grades/scores",
}


def load_scores(path):
    return {r["probe"]: r for r in map(json.loads, open(path))} if path.exists() else {}


rows = []
for run, arm, name, ks in READERS:
    for k in ks:
        for part, suf in (("main", ""), ("aux", "-aux")):
            fn = f"{arm}-k{k}{suf}.jsonl"
            now = load_scores(SCORE_NOW[run] / fn)
            pre = load_scores(SCORE_PRE[arm] / fn) if arm in SCORE_PRE else {}
            for p, r in now.items():
                rows.append({"group": name, "k": k, "part": part, "probe": p, "template": r["template"],
                             "score": r["score"], "score_pre": pre[p]["score"] if pre else r["score"]})
scores = pd.DataFrame(rows)

# C、D 在 k10、k20 上取 P 同一检查点的题
probes_at = {k: set(scores[(scores.group == "P") & (scores.k == k) & (scores.part == "main")].probe) for k in KS}
extra = []
for g in ("C", "D"):
    base = scores[(scores.group == g) & (scores.part == "main")]
    for k in (10, 20):
        extra.append(base[base.probe.isin(probes_at[k])].assign(k=k))
scores_all = pd.concat([scores] + extra, ignore_index=True)
ORDER = ["C", "D", "P", "A3", "A2", "A0", "A4", "A3F"]

main = scores_all[scores_all.part == "main"]
mean_now = main.pivot_table(index="k", columns="group", values="score", aggfunc="mean")[ORDER]
mean_pre = main.pivot_table(index="k", columns="group", values="score_pre", aggfunc="mean")[ORDER]
n_main = main.groupby(["group", "k"]).size().unstack(0)[ORDER]
print("主题集题数\n", n_main, "\n")
print("主题集平均分（现行计分）\n", mean_now.round(3), "\n")
print("四位小数\n", mean_now.round(4))

In [ ]:
aux = scores[scores.part == "aux"].pivot_table(index="k", columns="group", values="score", aggfunc="mean")
print("辅助题平均分（C、D 只有一次作答）\n", aux[[g for g in ORDER if g in aux]].round(3))

In [ ]:
changed = scores[(scores.score - scores.score_pre).abs() > 1e-9]
print("T3 修正前后分数不同的回答\n", changed[["group", "k", "part", "probe", "score_pre", "score"]], "\n")
diff = pd.DataFrame({"修正前": mean_pre.stack(), "修正后": mean_now.stack()})
print("平均分不同的格\n", diff[(diff["修正前"] - diff["修正后"]).abs() > 1e-9].round(4))

In [ ]:
k30m = scores[(scores.k == 30) & (scores.part == "main")]
counts = k30m.groupby("group").agg(full=("score", lambda s: int((s == 1).sum())), zero=("score", lambda s: int((s == 0).sum())),
                                   full_pre=("score_pre", lambda s: int((s == 1).sum())))
print("k30 满分与 0 分题数（full_pre 为修正前满分）\n", counts.loc[ORDER])
for (g, k) in [("A3", 20)]:
    s = scores[(scores.group == g) & (scores.k == k) & (scores.part == "main")]
    print(f"{g} k{k} 满分：修正前 {int((s.score_pre == 1).sum())}，修正后 {int((s.score == 1).sum())}")
t3 = k30m[k30m.template == "T3"].groupby("group")[["score_pre", "score"]].mean().round(3)
print("\nk30 T3 题型平均\n", t3.loc[ORDER])

### 2.1 配对差的点估计（修正前后）

同一组题上两组均分之差，等于逐题差的均值。只给点估计；区间见各实验记录（修正前算的区间没有重算）。

In [ ]:
def score_table(col):
    return main.pivot_table(index=["k", "probe"], columns="group", values=col)


PAIRS = [("D", "C", [30]), ("P", "C", KS), ("P", "D", KS), ("A3", "P", KS), ("A3", "D", KS), ("A3", "C", [20]),
         ("A0", "A3", KS), ("A0", "P", KS), ("A0", "D", KS), ("A0", "C", KS),
         ("A2", "A3", KS), ("A2", "A0", KS), ("A2", "P", KS), ("A2", "D", KS), ("A2", "C", KS),
         ("A4", "A3", KS), ("A4", "A0", KS), ("A4", "A2", KS), ("A4", "P", KS), ("A4", "D", KS), ("A4", "C", KS),
         ("A4", "A3F", [30]), ("A3F", "A3", KS), ("A3F", "A0", KS), ("A3F", "A2", KS), ("A3F", "P", KS)]
now_t, pre_t = score_table("score"), score_table("score_pre")
rows = []
for x, y, ks in PAIRS:
    for k in ks:
        a, b = now_t.loc[k], pre_t.loc[k]
        rows.append({"pair": f"{x} − {y}", "k": k, "修正前": (b[x] - b[y]).mean(), "修正后": (a[x] - a[y]).mean()})
pairdiff = pd.DataFrame(rows)
pairdiff["变化"] = (pairdiff["修正后"] - pairdiff["修正前"]).abs() > 5e-4
print(pairdiff.round(4).to_string(index=False))

### 2.2 溢出中断与失控

读者回答的结束状态（主题集与辅助题合计）。A4 的失控按退出码 `timeout` 计，其中一部分最后一条消息的结束原因也是 `error`；读者记录按退出码优先归类。

In [ ]:
r = readers.assign(status=readers.apply(lambda x: "timeout" if x.exit == "timeout" else ("overflow" if x.stop == "error" else "ok"), axis=1))
print(r.pivot_table(index=["group", "k"], columns="status", values="probe", aggfunc="count", fill_value=0))
print("\nA4 k30 的退出码与结束原因\n", readers[(readers.group == "A4") & (readers.k == 30)].groupby(["exit", "stop"]).size())

## 3 写者（建库），累计到检查点

各区间（k0→k10、k10→k20、k20→k30）求和后累计。A4 另列 Graphiti 内部 LLM 与会话后的等待（`settle_s`，不计入会话用时）。墙钟取自各写者记录“运行经过”的时刻（启动到检查点完成），以常量写入。

In [ ]:
WALL_WRITERS = {  # 由记录中的时刻相减；出处为各写者记录 §4
    "f2-A3": {10: "2:09", 20: "6:40", 30: "10:19"},   # f2/2026-10-21-f2-A3-writers.md §4
    "A3": {10: "2:14", 20: "6:08", 30: "10:57 + 重跑约 0:30"},  # f3 A3 写者记录 §4（原运行含原 seq 69 超时）
    "A2": {10: "2:11", 20: "6:05", 30: "9:35"},
    "A0": {10: "3:04", 20: "9:18", 30: "16:07"},
    "A4": {10: "7:24", 20: "19:27", 30: "30:47"},     # 含种子写入 41 分钟
}
wi = writers.groupby(["group", "k"])[RAW + G].sum()
wcum = add_costs(wi.groupby(level=0).cumsum().reset_index())
wcum["wall"] = [WALL_WRITERS[g][k] for g, k in zip(wcum.group, wcum.k)]
wcum["settle_h"] = (wcum.g_settle_s / 3600).round(2)
order = {g: i for i, g in enumerate(["f2-A3", "A3", "A2", "A0", "A4"])}
wcum = wcum.sort_values(["group", "k"], key=lambda s: s.map(order) if s.name == "group" else s).reset_index(drop=True)
cols = ["group", "k", "hours", "wall", "calls", "tools", "input", "cacheRead", "output", "T", "W_main", "W_sens"]
print(in_millions(wcum)[cols].to_string(index=False))

In [ ]:
g4 = in_millions(wcum[wcum.group == "A4"])
print(g4[["k", "g_calls", "g_prompt", "g_completion", "settle_h", "Tg", "Wg_main", "Wg_sens"]].to_string(index=False))

## 4 读者（各检查点与合计，含辅助题）

墙钟取自各读者记录“正式运行”表，以常量写入。C、D 为 12 路并发，原记录注明用时不进成本比较，只作参照。

In [ ]:
WALL_READERS = {  # 各读者记录的正式运行表；A3F 的合计由起止时刻 10-25 20:42 至 10-26 06:32 相减
    "P": {10: "0:43", 20: "1:03", 30: "8:02", "all": "9:49"},    # f2/…-ref1-P-originals.md §4
    "A3": {10: "0:54", 20: "1:14", 30: "8:57", "all": "11:04"},  # f3/read/…-A3-readers.md §3
    "A2": {10: "1:02", 20: "1:46", 30: "9:27", "all": "12:15"},  # f3/read/…-A2-readers.md §3
    "A0": {10: "0:47", 20: "1:09", 30: "6:23", "all": "8:19"},   # f3/read/…-A0-readers.md §3
    "A4": {10: "1:41", 20: "3:02", 30: "19:13", "all": "23:56"}, # f3/read/…-A4-readers.md §3
    "A3F": {10: "0:44", 20: "1:07", 30: "7:59", "all": "9:50"},  # f4/…-A3F-readers.md §3
}
rk = readers.groupby(["group", "k"])[RAW].sum()
rall = readers.groupby("group")[RAW].sum().assign(k="all").set_index("k", append=True)
rtab = add_costs(pd.concat([rk, rall]).reset_index())
rtab["cache_share"] = (rtab.cacheRead / rtab["T"]).round(3)
rtab["wall"] = [WALL_READERS.get(g, {}).get(k, "") for g, k in zip(rtab.group, rtab.k)]
order = {g: i for i, g in enumerate(COMPARED + ["C", "D"])}
rtab = rtab.sort_values(["group", "k"], key=lambda s: s.map(order) if s.name == "group" else s.astype(str)).reset_index(drop=True)
cols = ["group", "k", "hours", "wall", "calls", "tools", "input", "cacheRead", "output", "T", "W_main", "W_sens", "cache_share"]
print(in_millions(rtab)[cols].to_string(index=False))

In [ ]:
# A3F 读者的精确整数（原记录只给用时合计）与主题集每题平均（对照 A3F 读者记录 §4）
print(rtab[rtab.group == "A3F"][["k", "input", "cacheRead", "output"]].to_string(index=False))
m = readers[(readers.part == "main") & readers.group.isin(["A3", "A3F", "A0"])]
per_q = m.groupby(["group", "k"]).agg(minutes=("seconds", lambda s: s.mean() / 60), input_k=("input", lambda s: s.mean() / 1e3),
                                      cache_k=("cacheRead", lambda s: s.mean() / 1e3))
print(per_q.round(1))

## 5 建库加一轮读者

建库累计到 k30，加三个检查点的一轮读者（340 次会话）。P 没有建库；A3F 的建库即 A3 的建库（导出未计：重放 39 秒，派生字段提取没有计时，A3F 读者记录 §4、§8）。

In [ ]:
build = wcum[wcum.k == 30].set_index("group")
read = rtab[rtab.k == "all"].set_index("group")
rows = []
for g in COMPARED:
    b = build.loc[BUILD_OF[g]] if BUILD_OF[g] else pd.Series(0, index=build.columns)
    rd = read.loc[g]
    row = {"group": g, **{c: b[c] + rd[c] for c in RAW}}
    row.update({c: b.get(c, 0) for c in G})
    rows.append(row)
combo = add_costs(pd.DataFrame(rows).set_index("group"))
cols = ["hours", "calls", "tools", "input", "cacheRead", "output", "T", "W_main", "W_sens", "Tg", "Wg_main", "Wg_sens", "g_calls"]
print(in_millions(combo)[cols])

In [ ]:
ratio = (combo[["hours", "T", "Tg", "W_main", "Wg_main", "W_sens", "Wg_sens"]] / combo.loc["P", ["hours", "T", "Tg", "W_main", "Wg_main", "W_sens", "Wg_sens"]]).round(2)
print("相对 P 的倍数（Tg、Wg 为含 A4 内部 LLM 的值，其他组与 T、W 相同）\n", ratio)
diff_p = (combo[["hours", "calls", "input", "cacheRead", "output"]] - combo.loc["P", ["hours", "calls", "input", "cacheRead", "output"]])
print("\n相对 P 的差额（对照各读者记录“加上建库成本”一节）\n", diff_p.round(2))
split = pd.DataFrame({"build_W_main": [build.loc[BUILD_OF[g], "W_main"] if BUILD_OF[g] else 0 for g in COMPARED],
                      "build_Wg_main": [build.loc[BUILD_OF[g], "Wg_main"] if BUILD_OF[g] else 0 for g in COMPARED],
                      "read_W_main": [read.loc[g, "W_main"] for g in COMPARED]}, index=COMPARED)
print("\n加权成本中建库与读者的份额（M）\n", (split / 1e6).round(2))

## 6 三种口径的排序对照

由低到高。口径：总 token $T$、两种加权成本、会话用时之和；模型调用另列。写者 A4 的 $T$、$W$ 不含内部 LLM（含内部 LLM 时 A4 都排最后）。最后列出“总 token 更多而模型调用更少”的对子，看加权口径下是否变得更便宜。

In [ ]:
METRICS = {"T": "T", "W_main": "W_main", "W_sens": "W_sens", "用时": "seconds", "模型调用": "calls"}


def ranking(tab):
    out = {}
    for label, col in METRICS.items():
        s = tab[col].sort_values(kind="stable")
        parts = [str(s.index[0])]
        for prev, cur in zip(s.index[:-1], s.index[1:]):
            parts.append(("=" if s[cur] == s[prev] else "<") + " " + str(cur))
        out[label] = " ".join(parts)
    return out


scopes = {}
w30 = add_costs(wcum[wcum.k == 30].set_index("group"))
for k in KS:
    scopes[f"写者 k{k}"] = add_costs(wcum[wcum.k == k].set_index("group"))
rc = rtab[rtab.group.isin(COMPARED)]
for k in KS + ["all"]:
    scopes[f"读者 {k}"] = rc[rc.k == k].set_index("group")
combo_s = combo.assign(seconds=combo.hours * 3600)
scopes["建库加一轮读者"] = combo_s
rank = pd.DataFrame({name: ranking(t) for name, t in scopes.items()}).T
print(rank.to_string())

In [ ]:
rows = []
for name in ["写者 k30", "读者 all", "建库加一轮读者"]:
    t = scopes[name]
    for x in t.index:
        for y in t.index:
            if t.loc[x, "T"] > t.loc[y, "T"] and t.loc[x, "calls"] < t.loc[y, "calls"]:
                rows.append({"范围": name, "多 token 少调用": x, "对方": y,
                             "T": f"{t.loc[x, 'T'] / 1e6:.2f} > {t.loc[y, 'T'] / 1e6:.2f}",
                             "calls": f"{t.loc[x, 'calls']:.0f} < {t.loc[y, 'calls']:.0f}",
                             "W_main 变便宜": t.loc[x, "W_main"] < t.loc[y, "W_main"],
                             "W_sens 变便宜": t.loc[x, "W_sens"] < t.loc[y, "W_sens"]})
print(pd.DataFrame(rows).to_string(index=False))

## 7 按题型的成本（k30 主题集）

各题型的单题中位用时及相对 P 的倍数；题型族（跨论文 T3、T7、T8；单篇查找 T9、T10、T11、T14；其余）的每题平均及相对 P 的倍数。含溢出与失控的题。

In [ ]:
k30 = add_costs(readers[(readers.k == 30) & (readers.part == "main") & readers.group.isin(COMPARED)])
TYPES = ["T1", "T2", "T3", "T4", "T5", "T6", "T7", "T8", "T9", "T10", "T11", "T14"]
med = k30.pivot_table(index="template", columns="group", values="seconds", aggfunc="median").loc[TYPES, COMPARED]
n_t = k30[k30.group == "P"].groupby("template").size().loc[TYPES]
print("中位用时（秒）\n", med.round(0).assign(n=n_t))
print("\n相对 P 的倍数\n", med.div(med.P, axis=0).round(2).loc[CROSS + SINGLE])
print("\n中位工具调用\n", k30.pivot_table(index="template", columns="group", values="tools", aggfunc="median").loc[TYPES, COMPARED])
print("\n中位输出 token\n", k30.pivot_table(index="template", columns="group", values="output", aggfunc="median").loc[TYPES, COMPARED])

In [ ]:
fam = k30.assign(family=k30.template.map(lambda t: "cross" if t in CROSS else ("single" if t in SINGLE else "other")))
fm = fam.groupby(["family", "group"]).agg(n=("probe", "size"), minutes=("seconds", lambda s: s.mean() / 60),
                                          T_k=("T", lambda s: s.mean() / 1e3), W_main_k=("W_main", lambda s: s.mean() / 1e3),
                                          W_sens_k=("W_sens", lambda s: s.mean() / 1e3), calls=("calls", "mean"))
base = fm.xs("P", level="group")
rel = fm[["minutes", "T_k", "W_main_k", "W_sens_k", "calls"]].div(base[["minutes", "T_k", "W_main_k", "W_sens_k", "calls"]], level="family")
out = fm.round(1).join(rel.round(2), rsuffix="_xP")
print(out.loc[["cross", "single", "other"]].reindex(COMPARED, level="group").to_string())

## 8 回本轮数

把成本拆成一次性的建库 $B_g$ 与每轮读者 $R_g$，$n$ 轮之后的累计成本为 $B_g + n R_g$。P 没有建库（$B_P = 0$）。

- **相对 P 的回本轮数**：$n^{*} = B_g / (R_P - R_g)$，只在 $R_g < R_P$ 时存在；超过 $n^{*}$ 轮后该组的累计成本低于 P。
- **两组的交叉轮数**：$B_x + n R_x = B_y + n R_y$ 的解 $n^{\times} = (B_y - B_x) / (R_x - R_y)$，只在为正时存在；少于 $n^{\times}$ 轮时建库便宜的一组累计成本更低，多于时每轮便宜的一组更低。

**假设**（只说明成本结构，不是结论）：每轮读者是同一批题、同样的题型构成，成本与本次测得的一轮相同；库建一次后不再增长、不再维护；不计 A3F 的导出成本。主变体以“三个检查点的一轮读者”（340 次会话，与各读者记录的“加上建库成本”同一口径）为一轮；另给只读 k30 的变体（206 次会话）。轮数可以不是整数。

In [ ]:
def rounds(B, R, metric):
    rows = []
    for g in COMPARED:
        if g == "P":
            continue
        gap = R.loc["P", metric] - R.loc[g, metric]
        rows.append({"group": g, "B": B.get(g, 0.0), "R": R.loc[g, metric], "R_P": R.loc["P", metric],
                     "n*": B.get(g, 0.0) / gap if gap > 0 else float("inf")})
    return pd.DataFrame(rows).set_index("group")


def cross(B, R, metric, x, y):
    dr = R.loc[x, metric] - R.loc[y, metric]
    n = (B[y] - B[x]) / dr if dr != 0 else float("inf")
    return n if n > 0 else float("nan")


def build_costs(metric, with_g=False):
    col = {"W_main": "Wg_main", "W_sens": "Wg_sens", "T": "Tg"}.get(metric, metric) if with_g else metric
    return {g: (build.loc[BUILD_OF[g], col] if BUILD_OF[g] else 0.0) for g in COMPARED}


def per_round(sel):
    t = add_costs(readers[sel & readers.group.isin(COMPARED)].groupby("group")[RAW].sum())
    return t


R_all = per_round(readers.k.isin(KS))
R_k30 = per_round(readers.k == 30)
for label, R in (("一轮 = 三个检查点（340 次会话）", R_all), ("一轮 = 只读 k30（206 次会话）", R_k30)):
    print(f"### {label}")
    for metric, scale, unit in (("W_main", 1e6, "M"), ("W_sens", 1e6, "M"), ("T", 1e6, "M"), ("hours", 1, "小时")):
        B = build_costs(metric)
        t = rounds(B, R, metric)
        t[["B", "R", "R_P"]] = (t[["B", "R", "R_P"]] / scale).round(2)
        if metric != "hours":
            Bg = build_costs(metric, with_g=True)
            t.loc["A4 含内部 LLM"] = [Bg["A4"] / scale, R.loc["A4", metric] / scale, R.loc["P", metric] / scale,
                                      Bg["A4"] / (R.loc["P", metric] - R.loc["A4", metric]) if R.loc["A4", metric] < R.loc["P", metric] else float("inf")]
            t[["B", "R", "R_P"]] = t[["B", "R", "R_P"]].round(2)
        t["n*"] = t["n*"].round(2)
        print(f"\n{metric}（{unit}）\n", t)
        xs = {f"{x} 对 {y}": cross(B, R, metric, x, y) for x, y in [("A3F", "A0"), ("A3", "A0"), ("A2", "A0"), ("A3F", "A3")]}
        print("交叉轮数：", {k: (round(v, 2) if v == v else "无") for k, v in xs.items()})
    print()

In [ ]:
# 主口径下的累计成本（M），一轮 = 三个检查点
B = build_costs("W_main")
cum = pd.DataFrame({g: [(B[g] + n * R_all.loc[g, "W_main"]) / 1e6 for n in range(0, 6)] for g in COMPARED},
                   index=pd.Index(range(0, 6), name="轮数"))
print(cum.round(2))